In [1]:
# Imports & Configuration
from pathlib import Path

import duckdb
import polars as pl
from great_tables import GT

# Change only this path
SOURCE = Path("../../raw_data/raw_orders_cz.csv").resolve().as_posix()

# DuckDB connection
con = duckdb.connect()

In [2]:
# ============================================================
# Functions
# ============================================================

def show_sql(query: str, max_rows: int = 100):
    """Execute DuckDB SQL and display results using Great Tables."""
    result = con.sql(query).limit(max_rows)

    df = pl.DataFrame(
        result.fetchall(),
        schema=result.columns,
        orient="row"
    )

    display(GT(df))

EXPECTED_CURRENCY = {
    "cz": "CZK",
    "sk": "EUR",
    "hu": "HUF"
}[Path(SOURCE).stem.split("_")[-1]]

In [3]:
# 01 | Source Overview

show_sql(f"""
    SUMMARIZE
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
""")

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
order_id,VARCHAR,CZ-100001,CZ-109264,9196,None,None,None,None,None,14709,0.00
line_id,VARCHAR,14520000000000,14520000284412,14419,None,None,None,None,None,14709,0.00
created_at,VARCHAR,2025-12-31T23:01:38Z,2026-09-30T23:59:08Z,9083,None,None,None,None,None,14709,0.00
sku,VARCHAR,FB-BOARD-L,FB-WOK-30-BLK,31,None,None,None,None,None,14709,0.00
quantity,VARCHAR,1,2,2,None,None,None,None,None,14709,0.00
price_with_vat,VARCHAR,1090.0,990.0,28,None,None,None,None,None,14709,0.00
discount_with_vat,VARCHAR,0.0,99.0,147,None,None,None,None,None,14709,0.00
currency,VARCHAR,CZK,CZK,1,None,None,None,None,None,14709,0.00
customer_id,VARCHAR,cz-000001,cz-006076,6097,None,None,None,None,None,14709,7.04
cancelled_at,VARCHAR,2026-01-02T03:46:08Z,2026-10-01T15:41:40Z,242,None,None,None,None,None,14709,97.41


In [4]:
# 02 | Data Quality Checks

show_sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_rows,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(order_id), '') IS NULL
               OR NULLIF(TRIM(line_id), '') IS NULL
               OR NULLIF(TRIM(sku), '') IS NULL
               OR NULLIF(TRIM(created_at), '') IS NULL
               OR NULLIF(TRIM(exported_at), '') IS NULL
        ) AS missing_required,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(customer_id), '') IS NULL
        ) AS missing_customer,

        COUNT(*) FILTER (
            WHERE cancelled_at IS NOT NULL
        ) AS cancelled_rows,

        COUNT(*) FILTER (
            WHERE TRY_CAST(quantity AS INTEGER) IS NULL
               OR TRY_CAST(quantity AS INTEGER) <= 0
               OR TRY_CAST(price_with_vat AS DECIMAL(18,2)) IS NULL
               OR TRY_CAST(price_with_vat AS DECIMAL(18,2)) < 0
               OR TRY_CAST(discount_with_vat AS DECIMAL(18,2)) IS NULL
               OR TRY_CAST(discount_with_vat AS DECIMAL(18,2)) < 0
        ) AS invalid_numeric,

        COUNT(*) FILTER (
            WHERE currency IS DISTINCT FROM '{EXPECTED_CURRENCY}'
        ) AS invalid_currency,

        COUNT(*) FILTER (
            WHERE TRY_CAST(created_at AS TIMESTAMPTZ) IS NULL
               OR TRY_CAST(exported_at AS TIMESTAMP) IS NULL
               OR (
                   NULLIF(TRIM(cancelled_at), '') IS NOT NULL
                   AND TRY_CAST(cancelled_at AS TIMESTAMPTZ) IS NULL
               )
        ) AS invalid_timestamps

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

total_rows,duplicate_rows,missing_required,missing_customer,cancelled_rows,invalid_numeric,invalid_currency,invalid_timestamps
14709,610,0,1035,381,0,0,0


In [5]:
# 03 | Duplicate Investigation

show_sql(f"""
    WITH versions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY line_id
                ORDER BY exported_at DESC
            ) AS rn
        FROM read_csv(
            '{SOURCE}',
            all_varchar = true
        )
    )
    SELECT
        COUNT(*) AS duplicated_lines,
        COUNT(*) FILTER (
            WHERE old.cancelled_at IS DISTINCT FROM new.cancelled_at
        ) AS changed_cancellations,
        COUNT(*) FILTER (
            WHERE old.quantity IS DISTINCT FROM new.quantity
               OR old.price_with_vat IS DISTINCT FROM new.price_with_vat
               OR old.discount_with_vat IS DISTINCT FROM new.discount_with_vat
        ) AS changed_financial_values
    FROM versions old
    JOIN versions new
        ON old.line_id = new.line_id
       AND old.rn = 2
       AND new.rn = 1
""")

duplicated_lines,changed_cancellations,changed_financial_values
610,272,0


In [6]:
# 04 | Business Validity

show_sql(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE TRIM(order_id) = ''
               OR TRIM(line_id) = ''
               OR TRIM(sku) = ''
               OR TRIM(currency) = ''
        ) AS empty_required,

        COUNT(*) FILTER (
            WHERE TRY_CAST(discount_with_vat AS DECIMAL(18,2))
                > TRY_CAST(price_with_vat AS DECIMAL(18,2))
                * TRY_CAST(quantity AS INTEGER)
        ) AS excessive_discount,

        COUNT(*) FILTER (
            WHERE cancelled_at IS NOT NULL
              AND TRY_CAST(cancelled_at AS TIMESTAMPTZ)
                < TRY_CAST(created_at AS TIMESTAMPTZ)
        ) AS invalid_cancellation,

        COUNT(DISTINCT order_id) AS unique_orders,
        COUNT(DISTINCT sku) AS unique_skus

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

empty_required,excessive_discount,invalid_cancellation,unique_orders,unique_skus
0,0,0,9264,32


In [7]:
# 05 | Schema

show_sql(f"""
    DESCRIBE
    SELECT *
    FROM read_csv_auto('{SOURCE}')
""")

column_name,column_type,null,key,default,extra
order_id,VARCHAR,YES,None,None,None
line_id,BIGINT,YES,None,None,None
created_at,TIMESTAMP WITH TIME ZONE,YES,None,None,None
sku,VARCHAR,YES,None,None,None
quantity,BIGINT,YES,None,None,None
price_with_vat,DOUBLE,YES,None,None,None
discount_with_vat,DOUBLE,YES,None,None,None
currency,VARCHAR,YES,None,None,None
customer_id,VARCHAR,YES,None,None,None
cancelled_at,TIMESTAMP WITH TIME ZONE,YES,None,None,None


In [8]:
# 06 | Sample Data

show_sql(f"""
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
    LIMIT 5
""")

order_id,line_id,created_at,sku,quantity,price_with_vat,discount_with_vat,currency,customer_id,cancelled_at,exported_at
CZ-100081,14520000002477,2025-12-31T23:13:01Z,FB-CASS-28,1,2690.0,404.0,CZK,cz-000008,None,2026-01-01 02:00:01
CZ-100055,14520000001711,2025-12-31T23:20:03Z,FB-MITT,1,290.0,44.0,CZK,cz-000044,None,2026-01-01 02:00:03
CZ-100055,14520000001728,2025-12-31T23:20:03Z,FB-LID-24,1,450.0,68.0,CZK,cz-000044,None,2026-01-01 02:00:03
CZ-100055,14520000001741,2025-12-31T23:20:03Z,FB-BOARD-L,1,790.0,118.0,CZK,cz-000044,None,2026-01-01 02:00:03
CZ-100102,14520000003252,2025-12-31T23:53:58Z,FB-PAN-24,1,1490.0,224.0,CZK,cz-000077,None,2026-01-01 02:00:04


### 07 | Findings & Staging Rules – CZ Orders

**Source:** `raw_orders_cz.csv`

#### Data Quality Summary

| Check | Result |
|---|---:|
| Total rows | 14,709 |
| Unique orders | 9,264 |
| Unique SKUs | 32 |
| Duplicate rows (`line_id`) | 610 |
| Unchanged duplicate records | 338 |
| Updated cancellations | 272 |
| Missing `customer_id` | 1,035 |
| Cancelled rows | 381 |
| Missing required fields | 0 |
| Invalid numeric values | 0 |
| Invalid currencies | 0 |
| Invalid timestamps | 0 |
| Invalid business values | 0 |

#### Findings & Staging Decisions

1. **Duplicates:** Keep the latest record per `line_id` using `exported_at DESC`. Duplicate exports include 272 cancellation updates.
2. **Missing customers:** Preserve NULL values in `customer_id`.
3. **Cancellations:** Preserve `cancelled_at`. Exclude cancelled lines from sales calculations in the business layer.
4. **Data types:** Use String for identifiers, Int64 for quantity, Decimal(18,2) for monetary values and Datetime for timestamps.
5. **Reporting date:** Preserve UTC timestamps in staging. Convert to `Europe/Prague` in the business layer.

**Expected staging output:** 14,099 unique order lines.

**Pending cross-source checks:** SKU integrity against `raw_products` and FX rate coverage.